## import libraries

In [56]:
import pandas as pd
from geotessera import GeoTessera
from pathlib import Path
import os

## read datasets

In [57]:
data_root = Path("_data/exports/crop_country_exports")

In [58]:
def get_inventory(root_path):
    inventory = []

    for file_path in root_path.glob("*.csv"):
        parts = file_path.stem.split('_')
        
        if len(parts) > 3:
            crop = "_".join(parts[:-2])
            country = parts[-2]
            year = int(parts[-1])

        else:  
            crop = parts[0]
            country = parts[-2]
            year = int(parts[-1])

        inventory.append({
            "crop": crop,
            "country": country,
            "year": year,
            "path": file_path
        })

    df = pd.DataFrame(inventory)

    if not df.empty:
        df = df.sort_values(by=["crop", 'year']).reset_index(drop=True)

    return df

In [59]:
available_datasets = get_inventory(data_root)
available_datasets

,crop,country,year,path
0,maize_corn_popcorn,ee,2016,_data/exports/crop_country_exports/maize_corn_...
1,maize_corn_popcorn,at,2016,_data/exports/crop_country_exports/maize_corn_...
2,maize_corn_popcorn,be,2016,_data/exports/crop_country_exports/maize_corn_...
3,maize_corn_popcorn,dk,2016,_data/exports/crop_country_exports/maize_corn_...
4,maize_corn_popcorn,bg,2016,_data/exports/crop_country_exports/maize_corn_...
5,maize_corn_popcorn,ie,2017,_data/exports/crop_country_exports/maize_corn_...
6,maize_corn_popcorn,pt,2017,_data/exports/crop_country_exports/maize_corn_...
7,maize_corn_popcorn,pt,2018,_data/exports/crop_country_exports/maize_corn_...
8,maize_corn_popcorn,bg,2018,_data/exports/crop_country_exports/maize_corn_...
9,maize_corn_popcorn,dk,2018,_data/exports/crop_country_exports/maize_corn_...


## filter dataframe

In [78]:
year = 2019
crop = 'winter_barley'

In [79]:
def filter_df(df, crop, year):
    return df[(df['crop'] == crop) & (df['year'] == year)]

In [80]:
filtered_dataset = filter_df(available_datasets, crop, year)
filtered_dataset

,crop,country,year,path
54,winter_barley,dk,2019,_data/exports/crop_country_exports/winter_barl...
55,winter_barley,bg,2019,_data/exports/crop_country_exports/winter_barl...
56,winter_barley,ee,2019,_data/exports/crop_country_exports/winter_barl...
57,winter_barley,at,2019,_data/exports/crop_country_exports/winter_barl...
58,winter_barley,ie,2019,_data/exports/crop_country_exports/winter_barl...
59,winter_barley,be,2019,_data/exports/crop_country_exports/winter_barl...


In [81]:
def create_dataset(df):
    country_dfs = {}

    for country, group in df.groupby('country'):
        country_dfs[country] = pd.concat([pd.read_csv(p) for p in group['path']], ignore_index=True)
    
    return country_dfs

In [82]:
dataset = create_dataset(filtered_dataset)
dataset.keys()

dict_keys(['at', 'be', 'bg', 'dk', 'ee', 'ie'])

## retrieve embeddings

In [83]:
gt = GeoTessera(embeddings_dir="_data/embeddings_dir")

In [84]:
def process_embeddings(gdf, country_id, year, include_metadate=True):
    coords = list(zip(gdf['long'], gdf['lat']))

    print(f"Extracting embeddings for {country_id}...")
    embeddings, metadata_list = gt.sample_embeddings_at_points(coords, 
                                                               year=year, 
                                                               include_metadata=include_metadate)
    print(f"Embeddings extracted for {country_id}!")
    return embeddings, metadata_list, coords, country_id

In [85]:
def emb_meta_to_df(embedding, metadata, coords, crop_name, country_id, year):
    print(f"Converting information to a dataframe for {country_id}...")
    
    safe_metadata = [m if m is not None else {} for m in metadata]
    df_row = pd.DataFrame(safe_metadata)

    safe_embeddings = []
    for e in embedding:
        if e is not None:
            safe_embeddings.append(e.flatten())
        else:
            safe_embeddings.append(None) 
    
    df_row['embedding'] = safe_embeddings
    df_row['long_lat'] = coords
    df_row['crop'] = crop_name
    df_row['country_id'] = country_id
    df_row['year'] = year

    # df_row = df_row.fillna("-")

    print(f"Information converted for {country_id}! Total rows: {len(df_row)}")
    return df_row

# after 
# oi = embeddings2.flatten()
# oi.reshape(1, 128).shape

In [86]:
# to avoid running everything at the same time and run out of memory
keys = ['at', 'be', 'bg']
dataset.keys()

dict_keys(['at', 'be', 'bg', 'dk', 'ee', 'ie'])

In [87]:
result = []

for key in keys:
# for key in dataset.keys():
    country = dataset[key]['country_id'][0]
    gdf = dataset[key]
    embd, meta, coords, country_id = process_embeddings(gdf, country, year)
    result_df = emb_meta_to_df(embd, meta, coords, crop, country_id, year)

    result.append(result_df)

Extracting embeddings for at...
Embeddings extracted for at!
Converting information to a dataframe for at...
Information converted for at! Total rows: 500
Extracting embeddings for be...
Embeddings extracted for be!
Converting information to a dataframe for be...
Information converted for be! Total rows: 500
Extracting embeddings for bg...
Embeddings extracted for bg!
Converting information to a dataframe for bg...
Information converted for bg! Total rows: 500


## export data

In [88]:
export_folder = "_data/exports/embeddings_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for df in result:
    country = df['country_id'].iloc[0]
    file_name = f"{crop}_{country}_{year}_embedding.parquet"
    full_path = os.path.join(export_folder, file_name)
    df.to_parquet(full_path, index=False, engine='pyarrow')
    print(f"Saved: {full_path}")

# To load it back later:
# df = pd.read_parquet('country_crop_embeddings.parquet')

Saved: _data/exports/embeddings_exports/winter_barley_at_2019_embedding.parquet
Saved: _data/exports/embeddings_exports/winter_barley_be_2019_embedding.parquet
Saved: _data/exports/embeddings_exports/winter_barley_bg_2019_embedding.parquet
